# Groove2Groove-PyTorch — Colab training

End-to-end training notebook. Workflow:

1. Mount Drive — checkpoints and submissions only.
2. Clone the GitHub repo.
3. Install Python dependencies.
4. Download the competition dataset directly to Colab's local SSD via `kagglehub`.
5. Smoke-test (50 steps, tiny model) to confirm the full pipeline.
6. Full training — checkpoints written straight to Drive.
7. Generate a submission CSV and copy to Drive.

**One-time Colab setup — Secrets (left sidebar → 🔑):**

| Secret | Value |
|---|---|
| `GITHUB_REPO` | Clone URL, e.g. `https://github.com/user/repo.git` |
| `KAGGLE_API_TOKEN` | New-style Kaggle token (starts with `KGAT_`) — `kaggle.com → Account → Settings → API → Create New Token` |

> **Note:** Accept the competition rules on kaggle.com before the first download.

After setting the two secrets once, run top-to-bottom without any edits.

> **GPU:** *Runtime → Change runtime type → T4 GPU* before starting.

In [ ]:
# === Training hyperparameters — edit only to customise the run ==============
GITHUB_BRANCH  = "main"
RUN_NAME       = "exp2"
DRIVE_RUNS_DIR = "/content/drive/MyDrive/g2g_runs"
KAGGLE_COMP    = "sapienza-genai-hackathon"
BATCH_SIZE     = 16
LR             = 5e-4
MAX_STEPS      = 150_000
NUM_WORKERS    = 2
# ============================================================================

# Secrets — read from Colab sidebar, no edits needed below
from google.colab import userdata
import re, os

GITHUB_REPO = userdata.get("GITHUB_REPO")
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

_repo_name = re.sub(r'\.git$', '', GITHUB_REPO.rstrip('/').split('/')[-1])
CODE_DIR   = f"/content/{_repo_name}"
LOGDIR     = f"{DRIVE_RUNS_DIR}/{RUN_NAME}"
print(f"CODE_DIR = {CODE_DIR}")
print(f"LOGDIR   = {LOGDIR}")

## 0. Sanity-check the runtime

In [ ]:
!nvidia-smi || echo 'No GPU — switch to Runtime → Change runtime type → GPU.'
import torch
print("torch:", torch.__version__,
      "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 1. Mount Drive

Only used to persist checkpoints and submissions. The dataset lives on local SSD.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(LOGDIR, exist_ok=True)
print("Checkpoint dir:", LOGDIR)

## 2. Clone the repo

Uses the `GITHUB_REPO` secret. For a private repo include a personal access token:
`https://<token>@github.com/user/repo.git` — do **not** commit the token anywhere.

In [ ]:
import os

if os.path.isdir(CODE_DIR):
    print("Repo already cloned — pulling latest...")
    !cd {CODE_DIR} && git fetch --depth=1 origin {GITHUB_BRANCH} && git reset --hard origin/{GITHUB_BRANCH}
else:
    !git clone --depth=1 -b {GITHUB_BRANCH} {GITHUB_REPO} {CODE_DIR}

!ls {CODE_DIR}

# utility/ is now a sub-package inside g2g_pytorch/ — verify the structure
assert os.path.isdir(f"{CODE_DIR}/g2g_pytorch"), \
    f"g2g_pytorch not found under {CODE_DIR}. Check GITHUB_REPO."
assert os.path.isdir(f"{CODE_DIR}/g2g_pytorch/utility"), \
    f"utility sub-package not found at {CODE_DIR}/g2g_pytorch/utility."
print("Repo structure OK.")

## 3. Install dependencies

`torch`, `numpy`, and `pandas` are pre-installed on Colab. We only add `kagglehub` and `pretty_midi`.

In [ ]:
!pip install -q kagglehub pretty_midi
import numpy, pandas, torch, pretty_midi, kagglehub
print("numpy", numpy.__version__,
      "| pandas", pandas.__version__,
      "| torch", torch.__version__,
      "| pretty_midi", pretty_midi.__version__,
      "| kagglehub", kagglehub.__version__)

## 4. Download dataset from Kaggle

`kagglehub` downloads to Colab's local SSD and caches the result — re-runs skip the download.

> Accept the competition rules on kaggle.com before running this cell the first time.

In [ ]:
import kagglehub, os, csv

path = kagglehub.competition_download(KAGGLE_COMP)
print("Downloaded to:", path)

# manifest.csv may sit inside a version subfolder
candidates = []
for root, subdirs, files in os.walk(path):
    if "manifest.csv" in files:
        candidates.append(os.path.join(root, "manifest.csv"))
        break

assert candidates, f"manifest.csv not found under {path}"
DATASET_DIR = os.path.dirname(candidates[0])
print("DATASET_DIR =", DATASET_DIR)

rolls_dir    = os.path.join(DATASET_DIR, "rolls")
profiles_dir = os.path.join(DATASET_DIR, "style_profiles")
print(f"style folders : {len(os.listdir(rolls_dir)) if os.path.isdir(rolls_dir) else 0}")
print(f"style profiles: {len(os.listdir(profiles_dir)) if os.path.isdir(profiles_dir) else 0}")

# Auto-detect which split name the competition uses for test items
with open(f"{DATASET_DIR}/manifest.csv") as _f:
    _splits = {r['split'] for r in csv.DictReader(_f)}
print("Splits in manifest:", _splits)
TEST_SPLIT = next((s for s in ("test_public", "test") if s in _splits), list(_splits)[0])
print("Using test split:", TEST_SPLIT)

## 5. Smoke test (~1 min)

Tiny model, 50 steps. Confirms dataset path, GPU, and val scorer all work before a long run.

In [ ]:
!cd {CODE_DIR} && python -m g2g_pytorch.train \
    --debug \
    --dataset-root {DATASET_DIR} \
    --logdir /content/runs/smoke \
    --num-workers 0

## 6. Full training

Checkpoints land in `LOGDIR` on Drive and survive runtime resets.
If the session disconnects, re-run cells 0–4 to restore all variables, then use the **Resume** cell below.

In [ ]:
!cd {CODE_DIR} && python -m g2g_pytorch.train \
    --dataset-root {DATASET_DIR} \
    --logdir       {LOGDIR} \
    --batch-size   {BATCH_SIZE} \
    --lr           {LR} \
    --max-steps    {MAX_STEPS} \
    --num-workers  {NUM_WORKERS}

### Resuming after a disconnect

Re-run cells 0–4 to restore all variables, then uncomment and run the cell below.

In [ ]:
# !cd {CODE_DIR} && python -m g2g_pytorch.train \
#     --dataset-root {DATASET_DIR} \
#     --logdir       {LOGDIR} \
#     --batch-size   {BATCH_SIZE} \
#     --lr           {LR} \
#     --max-steps    {MAX_STEPS} \
#     --num-workers  {NUM_WORKERS} \
#     --resume       {LOGDIR}/model.pt

## 7. Generate submission CSV

Runs the trained model on the test split and writes the Kaggle format (`ID, notes`). Result is copied to Drive.

In [ ]:
SUBMISSION_LOCAL = f"/content/submission_{RUN_NAME}.csv"
SUBMISSION_DRIVE = f"{LOGDIR}/submission.csv"

!cd {CODE_DIR} && python -m g2g_pytorch.infer \
    --dataset-root {DATASET_DIR} \
    --ckpt         {LOGDIR}/model.pt \
    --output       {SUBMISSION_LOCAL} \
    --split        {TEST_SPLIT} \
    --batch-size   {BATCH_SIZE}

import shutil, os
shutil.copy(SUBMISSION_LOCAL, SUBMISSION_DRIVE)
print("Saved to Drive:", SUBMISSION_DRIVE,
      f"({round(os.path.getsize(SUBMISSION_DRIVE)/1e6, 2)} MB)")

## 8. Spot-check the submission

Every test item should appear with at least one note row.

In [ ]:
import pandas as pd, csv

sub = pd.read_csv(SUBMISSION_LOCAL)
print("rows:", len(sub), "| unique IDs:", sub['ID'].nunique())
print("avg notes/item:", sub['notes'].str.count(';').add(1).mean().round(1))
print(sub.head(3))

with open(f"{DATASET_DIR}/manifest.csv") as f:
    expected = {r['item_id'] for r in csv.DictReader(f) if r['split'] == TEST_SPLIT}
missing = expected - set(sub['ID'])
extra   = set(sub['ID']) - expected
print(f"\nExpected {len(expected)} test items | missing: {len(missing)} | extra: {len(extra)}")
assert not missing, f"Submission missing {len(missing)} items, e.g. {sorted(missing)[:5]}"
print("Submission looks good!")